# Proof: RocmPilot-patched nanoGPT runs on AMD
Clones nanoGPT, applies RocmPilot's device fix (`device='cuda'` -> guarded), and
runs nanoGPT's OWN sampler on the AMD GPU — generating real text. This is the
'it actually runs on AMD, not just a report' moment. Run in an AMD ROCm PyTorch Lab.

In [ ]:
import os, re, subprocess, sys

# 1) Get nanoGPT (public, no auth).
if not os.path.isdir('nanoGPT'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/karpathy/nanoGPT'], check=True)
os.chdir('nanoGPT')

# 2) Apply RocmPilot's device patch to sample.py (device='cuda' -> guarded lookup).
src = open('sample.py').read()
src = re.sub(r"""device\s*=\s*['\"]cuda['\"]""",
             'device = "cuda" if torch.cuda.is_available() else "cpu"', src, count=1)
open('sample.py', 'w').write(src)
print('patched:', [l for l in src.splitlines() if 'is_available' in l][:1])

# 3) nanoGPT deps (torch is already the ROCm build).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy', 'transformers', 'tiktoken'], check=True)

In [ ]:
# 4) Run nanoGPT's own sampler ON THE AMD GPU — downloads GPT-2 weights, generates text.
#    float32 for maximum reliability; --device=cuda runs on the Radeon via ROCm/HIP.
print('=== nanoGPT generating on AMD (Radeon / ROCm) ===\n')
subprocess.run([sys.executable, 'sample.py',
                '--init_from=gpt2', '--device=cuda',
                '--num_samples=1', '--max_new_tokens=60',
                '--dtype=float32'], check=True)